# 15 — チューニング実験室

症状→仮説→1群変更→同一条件比較という順序を、簡単なMPCで練習します。

**前提**: `14_logging_and_diagnosis.ipynb`

> 読み方: 「直感 → 数式 → 上流コード → 小実験 → 解釈」の順です。
> `実装事実` と書いた箇所は現行 `external/Quadruped-PyMPC` のコード、
> `学習用モデル` は理解のために単純化した再実装です。

In [1]:
from pathlib import Path
import os, sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebook_pympc":
    ROOT = ROOT.parent
PYMPC_ROOT = ROOT / "external" / "Quadruped-PyMPC"
assert PYMPC_ROOT.exists(), f"Quadruped-PyMPC が見つかりません: {PYMPC_ROOT}"
if str(PYMPC_ROOT) not in sys.path:
    sys.path.insert(0, str(PYMPC_ROOT))

os.environ.setdefault("ACADOS_SOURCE_DIR", str(PYMPC_ROOT / "quadruped_pympc" / "acados"))
os.environ.setdefault("MUJOCO_GL", "egl")
print("workspace :", ROOT)
print("PyMPC root:", PYMPC_ROOT)

workspace : /home/takuya/work/mpc_dog
PyMPC root: /home/takuya/work/mpc_dog/external/Quadruped-PyMPC


## 推奨する調整順

1. model/単位/frame/脚順の誤りを除外
2. 実摩擦以下のMPC摩擦、GRF・torque上限を確認
3. gaitの周波数・duty・swing heightで運動学的余裕を作る
4. footholdと速度指令を確認
5. Q/Rを状態群ごとに変更
6. horizon、dt、solver option
7. 高度機能（積分器、foothold最適化、RTI等）

重みから始めると、構造上不可能な歩容を大きな力で追わせる危険があります。

In [2]:
import numpy as np
from scipy.optimize import minimize
import pandas as pd

def trial(q, r, N=10):
    x, target, sat = -1.0, 1.0, 0
    sqerr, effort = 0., 0.
    for _ in range(20):
        def cost(u):
            xs = x + np.concatenate([[0.], np.cumsum(u)])
            return q*np.sum((xs-target)**2) + r*np.sum(u**2)
        sol = minimize(cost, np.zeros(N), bounds=[(-.3,.3)]*N).x
        u = sol[0]
        sat += abs(u) > .299
        x += u; sqerr += (x-target)**2; effort += u*u
    return np.sqrt(sqerr/20), effort, sat/20

rows = []
for q in [1, 10, 100]:
    for r in [0.1, 1, 10]:
        rmse, effort, sat = trial(q, r)
        rows.append({"Q":q, "R":r, "RMSE":rmse, "effort":effort, "sat_rate":sat})
pd.DataFrame(rows).round(3)

,Q,R,RMSE,effort,sat_rate
0,1,0.1,0.591,0.574,0.30
1,1,1.0,0.591,0.558,0.30
2,1,10.0,0.610,0.458,0.15
3,10,0.1,0.591,0.579,0.30
4,10,1.0,0.591,0.574,0.30
5,10,10.0,0.591,0.558,0.30
6,100,0.1,0.591,0.580,0.30
7,100,1.0,0.591,0.579,0.30
8,100,10.0,0.591,0.574,0.30


Qを増やしてRMSEが下がっても、飽和率と入力energyが悪化することがあります。
目的関数値だけでなくPlant側の制約指標を併記してください。

実験記録には「変更理由」「期待する向き」「副作用」「採否」を残します。
1 trialに複数群を変えると、改善の原因を学べません。

## 章末チェック

出力を眺めるだけでなく、次を自分の言葉で答えてください。

1. この章の入力・出力の shape、単位、座標系は何か。
2. 変更可能な量と、他の章から渡される量は何か。
3. パラメータを2倍にしたとき、どのグラフがどちらへ変化するか。
4. 現行実装の事実と、学習用の近似を区別できるか。